In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# AWARE-RG Lattice Pipeline

This cleaned notebook builds the AWARE-RG abstraction lattice from PDDL predicates.

Main stages:
1. Parse predicates from the PDDL domain.
2. Map predicates to MITRE ATT&CK for ICS labels.
3. Build an FCA-style context matrix.
4. Create IT and OT abstraction nodes.
5. Add bridge nodes.
6. Export the final lattice JSON.


In [26]:
# =========================
# User configuration
# =========================

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation"

DOMAIN_PDDL = BASE_DIR + "/model/FULL/domain.pddl"
PROBLEM_PDDL = BASE_DIR + "/model/FULL/problem.pddl"
MAPPING_CSV = BASE_DIR + "/pddl_mitre_ics_mapping.csv"
FILTERED_NODE_DIR = BASE_DIR + "/filtered_node_csv"
LATTICE_OUTPUT_DIR = BASE_DIR + "/lattice_output"

OUTPUT_BASE_DIR = BASE_DIR + "/model"


## 1. Install dependencies


In [3]:
!pip install -U bitsandbytes>=0.46.1

## 2. Parse PDDL predicates


In [27]:
import re
from pathlib import Path


def remove_comments(pddl_text: str) -> str:
    """
    Remove PDDL comments.
    Anything after ';' on a line is treated as a comment.
    """
    lines = []
    for line in pddl_text.splitlines():
        line = line.split(";")[0]
        lines.append(line)
    return "\n".join(lines)


def find_balanced_section(text: str, start_index: int) -> str:
    """
    Given the index of an opening parenthesis, return the full balanced
    parenthesized section.
    """
    depth = 0
    for i in range(start_index, len(text)):
        if text[i] == "(":
            depth += 1
        elif text[i] == ")":
            depth -= 1

        if depth == 0:
            return text[start_index:i + 1]

    raise ValueError("Unbalanced parentheses in PDDL file.")


def extract_predicates_section(pddl_text: str) -> str:
    """
    Extract the full (:predicates ...) section from a PDDL domain file.
    """
    text = remove_comments(pddl_text)

    match = re.search(r"\(:predicates\b", text, re.IGNORECASE)
    if not match:
        raise ValueError("No (:predicates ...) section found.")

    start_index = match.start()
    return find_balanced_section(text, start_index)


def extract_predicates_from_section(predicates_section: str) -> list[str]:
    """
    Extract individual predicate definitions from the predicates section.
    """
    # Remove the outer (:predicates ... )
    inner = re.sub(
        r"^\s*\(:predicates\b",
        "",
        predicates_section.strip(),
        flags=re.IGNORECASE
    ).strip()

    if inner.endswith(")"):
        inner = inner[:-1].strip()

    predicates = []
    i = 0

    while i < len(inner):
        if inner[i] == "(":
            predicate = find_balanced_section(inner, i)
            predicates.append(predicate.strip())
            i += len(predicate)
        else:
            i += 1

    return predicates


def extract_predicates_from_domain(domain_file: str) -> list[str]:
    """
    Main function: read domain.pddl and return all predicates.
    """
    pddl_text = Path(domain_file).read_text(encoding="utf-8")

    predicates_section = extract_predicates_section(pddl_text)
    predicates = extract_predicates_from_section(predicates_section)

    return predicates

In [28]:
import pandas as pd

predicates = extract_predicates_from_domain(DOMAIN_PDDL)

df_extracted_predicates = pd.DataFrame({
    "pddl_item": predicates
})

df_extracted_predicates.to_csv(BASE_DIR+"extracted_predicates.csv", index=False)

df_extracted_predicates["pddl_item"]


,pddl_item
0,(has-compromised-customer-pc)
1,(has-compromised-engineering-workstations)
2,(has-connected ?at - node ?to - node)
3,(has-access-to-windows-server ?node - node)
4,(has-vulnerability-CVE-2019-0575 ?node - node)
5,(has-vulnerability-CVE-2019-0584 ?node - node)
6,(has-vulnerability-CVE-2018-0538 ?node - node)
7,(has-done-remote-code-execution ?node - node)
8,(has-improper-vpn-firewall-configuration ?node - node)
9,(has-vulnerability-CVE-2018-0296 ?node - node)


## 3. Map predicates to MITRE ATT&CK for ICS


In [29]:
# ── 1. Imports ───────────────────────────────────────────────
import torch
import pandas as pd
import json
import re
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
)


In [7]:
# ── 2. Model Initialisation ──────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (4-bit) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,        # greedy -- most stable for JSON output
    return_full_text=False,
)

Loading tokenizer ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
# ── 3. Label Definitions ─────────────────────────────────────
# Full MITRE ATT&CK for ICS tactic list (do not edit).
MITRE_ICS_TACTICS = [
    "Initial Access",
    "Execution",
    "Persistence",
    "Privilege Escalation",
    "Evasion",
    "Discovery",
    "Lateral Movement",
    "Collection",
    "Command and Control",
    "Inhibit Response Function",
    "Impair Process Control",
    "Impact",
]

In [30]:
# ── 4. Domain Configuration ──────────────────────────────────
# +----------------------------------------------------------+
# |  EDIT THIS SECTION FOR YOUR CPS DOMAIN                   |
# +----------------------------------------------------------+

CPS_DOMAIN_DESCRIPTION = (
    "An oil-and-gas flare system in which a cyber attacker compromises IT "
    "endpoints, pivots through the corporate and OT networks, exploits "
    "Windows Server and PLC vulnerabilities to execute remote code, "
    "performs a denial-of-service attack on a safety PLC, and triggers a "
    "physical fault-propagation chain that ultimately causes a flare "
    "flameout -- a system-wide safety hazard."
)

PDDL_ITEMS = df_extracted_predicates["pddl_item"].dropna().astype(str).tolist()

print(f"Loaded {len(PDDL_ITEMS)} PDDL predicates.")

Loaded 22 PDDL predicates.


In [31]:
# ── 5. Optional Ground Truth ──────────────────────────────────
# Ground truth compares only the primary label.
# Format:
#   { "pddl predicate": "expected MITRE ICS tactic" }

KNOWN_CORRECT: dict[str, str] = {
    # Initial Access / Foothold
    "(has-compromised-customer-pc)": "Initial Access",
    "(has-improper-vpn-firewall-configuration ?node - node)": "Initial Access",
    "(has-compromised-engineering-workstations)": "Initial Access",


    # Execution / Exploitation
    "(has-vulnerability-CVE-2019-0575 ?node - node)": "Execution",
    "(has-vulnerability-CVE-2019-0584 ?node - node)": "Execution",
    "(has-vulnerability-CVE-2018-0538 ?node - node)": "Execution",
    "(has-vulnerability-CVE-2017-9312 ?node - node)": "Execution",
    "(has-exploit-improperly-implemented-fields-in-stack ?node - node)": "Execution",
    "(has-done-remote-code-execution ?node - node)": "Execution",
    "(has-vulnerability-CVE-2018-0296 ?node - node)": "Execution",

    # Lateral Movement / Network Traversal
    "(has-connected ?at - node ?to - node)": "Lateral Movement",
    "(has-access-to-windows-server ?node - node)": "Lateral Movement",
    "(has-access-to-network ?node - node)": "Lateral Movement",

    # Discovery / OT Reconnaissance
    "(has-access-to-plc-port-1132-tcp ?node - node)": "Lateral Movement",
    "(has-Allen-Bradley-L30ERMS-safety-devices-v30-and-earlier ?node - node)": "Discovery",

    # ICS Digital Attack
    "(has-done-denial-of-service-in-PLC1 ?node - node)": "Impair Process Control",
    "(plc-offline ?node - node)": "Impair Process Control",

    # ICS Physical Consequences
    "(has-fault-valve-blocked-close-due-to-compromised ?node - node)": "Impair Process Control",
    "(has-fault-manual-isolation-valve-close-due-to-compromised ?node - node)": "Impair Process Control",
    "(has-fault-pilot-low-supply-pressure-due-to-compromised ?node - node)": "Impair Process Control",
    "(has-fault-pilot-extinction-due-to-compromised ?node - node)": "Impair Process Control",
    "(has-fault-flare-flameout)": "Impact",
}

In [32]:
# ── 7. Prompt Construction ───────────────────────────────────

_RULES = """\
TASK
====
You are a CPS security analyst. Map a single PDDL predicate from an
industrial control system security model to MITRE ATT&CK for ICS tactics.

RULES
=====
1. Expand the predicate into ONE plain-English sentence that describes
   what the predicate represents in the cyber/physical system.
2. Choose exactly ONE primary MITRE ATT&CK for ICS tactic.
3. Optionally choose zero to three secondary MITRE ATT&CK for ICS tactics.
4. The primary label should describe the predicate's main role in the PDDL model.
5. Secondary labels should capture other plausible MITRE interpretations.
6. Do not use NIST SP 1500-201.
7. Do not invent labels.
8. Do not return the primary label again as a secondary label.
9. Output valid JSON ONLY -- no prose before or after the JSON object.

PRIMARY VS SECONDARY LABELS
===========================
The primary label is the best single label for formal analysis, such as FCA.
The secondary labels preserve ambiguity when a predicate could reasonably map
to more than one MITRE tactic.

For example:
- A reachable PLC port can be primary Lateral Movement and secondary Discovery.
- A known device or firmware version can be primary Discovery and secondary Execution
  if it gates an exploit.
- A CVE predicate can be primary Execution and secondary Discovery.
- A PLC denial-of-service predicate can be primary Impair Process Control and
  secondary Inhibit Response Function or Impact.
- A final physical hazard is primary Impact.

DECISION TREE FOR PRIMARY LABEL
===============================
Work through these questions in order. Stop at the first YES.

Q1. Is this predicate a pre-existing compromised asset, exposed condition,
    misconfiguration, vulnerability, or access condition that gives the attacker
    an entry point into the CPS/ICS environment?
    YES -> primary_label = Initial Access

Q2. Does this predicate describe the attacker moving between network zones,
    reaching a remote host, reaching a server, reaching a PLC service port,
    or obtaining network reachability without yet executing an exploit?
    Examples: has-connected, has-access-to-network, has-access-to-server,
    has-access-to-plc-port, VPN/firewall traversal.
    YES -> primary_label = Lateral Movement

Q3. Does this predicate describe learning about OT assets, device models,
    firmware versions, exposed services, controller type, protocol availability,
    or other information used to understand the ICS environment?
    YES -> primary_label = Discovery

Q4. Does this predicate describe a specific exploitable vulnerability,
    CVE, exploit precondition, protocol stack flaw, or the outcome of
    successfully executing unauthorized code or commands on a target?
    YES -> primary_label = Execution

Q5. Does this predicate describe disabling, suppressing, degrading, or bypassing
    a safety system, alarm, interlock, protection function, or response capability?
    YES -> primary_label = Inhibit Response Function

Q6. Does this predicate describe direct disruption or manipulation of a controller,
    PLC, RTU, DCS, actuator behavior, control logic, physical process variable,
    or intermediate fault caused by the compromised control system?
    This includes PLC offline, denial of service on a controller, valve faults,
    pressure faults, pilot faults, actuator faults, or physical process degradation
    before the final system-wide consequence.
    YES -> primary_label = Impair Process Control

Q7. Does this predicate describe the final system-wide physical consequence,
    operational failure, outage, hazardous condition, damage, or terminal
    attack objective?
    Examples: flare flameout, blackout, explosion, toxic release,
    equipment destruction, production halt.
    YES -> primary_label = Impact

Q8. If none of the above fits, choose the closest MITRE ATT&CK for ICS tactic
    from the allowed list and justify why.

SECONDARY LABEL GUIDANCE
========================
After choosing the primary label, add secondary labels only when they are
semantically plausible.

Use these common secondary mappings:

- Initial Access may have secondary Lateral Movement if the predicate also
  enables movement into another network zone.

- Lateral Movement may have secondary Discovery when the predicate reveals
  reachable OT assets, ports, services, or network topology.

- Discovery may have secondary Execution when the discovered property is also
  an exploit precondition, such as a vulnerable firmware version or device model.

- Execution may have secondary Discovery when the predicate is a known CVE or
  vulnerability that must first be identified.

- Inhibit Response Function may have secondary Impair Process Control when
  disabling a protection or response function also changes process control.

- Impair Process Control may have secondary Inhibit Response Function when the
  disrupted controller is part of a safety or response function.

- Impair Process Control may have secondary Impact when the predicate represents
  serious physical degradation but is not the final terminal consequence.

- Impact should usually have no secondary label unless the final hazard is also
  explicitly described as process manipulation.

BOUNDARY RULES
==============
- Access to a port, host, network, or service is primarily Lateral Movement
  unless the predicate explicitly says the attacker executed code or exploited it.
- A reachable PLC service port may have secondary Discovery.
- A CVE or vulnerability predicate is primarily Execution when it is used as an
  exploit precondition.
- A CVE or vulnerability predicate may have secondary Discovery because it also
  represents knowledge of a vulnerable condition.
- Device model or firmware information is primarily Discovery if it identifies
  the target, but may have secondary Execution if it gates an exploit.
- PLC offline or denial of service on a controller is primarily Impair Process Control.
- Denial of service against a safety PLC may have secondary Inhibit Response Function.
- Intermediate physical faults caused by compromised control, such as valve
  blocked close, low pilot supply pressure, or pilot extinction, are primarily
  Impair Process Control.
- The final system-wide hazard or operational consequence is primarily Impact.
"""

In [33]:
_DOMAIN_CONTEXT_TEMPLATE = """\
CPS DOMAIN CONTEXT
==================
{description}

Use this context to interpret domain-specific predicate names.
The decision tree above takes precedence over surface-level keyword matching.
"""

In [35]:
_CROSS_DOMAIN_EXAMPLES: list[dict] = [
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-compromised-customer-pc)",
        "natural_language_expansion":
            "A customer-side PC has already been compromised, giving the attacker an entry point into the environment.",
        "primary_label": "Initial Access",
        "secondary_labels": [],
        "justification":
            "Q1: the predicate represents a pre-existing foothold used to enter the CPS/ICS environment."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-improper-vpn-firewall-configuration ?node - node)",
        "natural_language_expansion":
            "A VPN or firewall misconfiguration allows unauthorized access into a protected network zone.",
        "primary_label": "Initial Access",
        "secondary_labels": ["Lateral Movement"],
        "justification":
            "Q1: the predicate represents an exposed entry condition. It also supports Lateral Movement because it enables traversal across a network boundary."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-connected ?at - node ?to - node)",
        "natural_language_expansion":
            "There is a network connection that allows movement from one node to another.",
        "primary_label": "Lateral Movement",
        "secondary_labels": ["Discovery"],
        "justification":
            "Q2: the predicate represents network traversal. It may also support Discovery because it exposes topology."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-access-to-windows-server ?node - node)",
        "natural_language_expansion":
            "The attacker has access to a Windows Server from another compromised or reachable node.",
        "primary_label": "Lateral Movement",
        "secondary_labels": [],
        "justification":
            "Q2: the predicate represents movement or reachability to another host."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-access-to-plc-port-1132-tcp ?node - node)",
        "natural_language_expansion":
            "The attacker can reach TCP port 1132 on a PLC, exposing a controller service for further interaction.",
        "primary_label": "Lateral Movement",
        "secondary_labels": ["Discovery"],
        "justification":
            "Q2: the predicate primarily represents reachability to an OT asset. It also supports Discovery because it reveals a reachable PLC service port."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-Allen-Bradley-L30ERMS-safety-devices-v30-and-earlier ?node - node)",
        "natural_language_expansion":
            "The target is identified as an Allen-Bradley L30ERMS safety device running version 30 or earlier.",
        "primary_label": "Discovery",
        "secondary_labels": ["Execution"],
        "justification":
            "Q3: the predicate identifies device model and version. It has secondary Execution because the device/version condition can gate exploitability."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-vulnerability-CVE-2019-0575 ?node - node)",
        "natural_language_expansion":
            "The target node has CVE-2019-0575, a specific exploitable vulnerability.",
        "primary_label": "Execution",
        "secondary_labels": ["Discovery"],
        "justification":
            "Q4: the predicate primarily represents an exploit precondition. It also implies Discovery because the vulnerable condition has been identified."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-done-remote-code-execution ?node - node)",
        "natural_language_expansion":
            "The attacker has successfully executed unauthorized code on the target node.",
        "primary_label": "Execution",
        "secondary_labels": [],
        "justification":
            "Q4: the predicate represents successful unauthorized code execution."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-done-denial-of-service-in-PLC1 ?node - node)",
        "natural_language_expansion":
            "The attacker has performed a denial-of-service attack against PLC1, disrupting controller availability.",
        "primary_label": "Impair Process Control",
        "secondary_labels": ["Inhibit Response Function", "Impact"],
        "justification":
            "Q6: the predicate primarily disrupts PLC control. It may also inhibit response if PLC1 supports safety functions and may contribute to operational impact."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(plc-offline ?node - node)",
        "natural_language_expansion":
            "The PLC is offline and can no longer provide normal control authority over the process.",
        "primary_label": "Impair Process Control",
        "secondary_labels": ["Impact"],
        "justification":
            "Q6: the predicate represents direct loss of process control. It may also contribute to operational impact."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-fault-valve-blocked-close-due-to-compromised ?node - node)",
        "natural_language_expansion":
            "A valve becomes blocked closed as a physical process consequence of the compromised control system.",
        "primary_label": "Impair Process Control",
        "secondary_labels": ["Impact"],
        "justification":
            "Q6: the predicate represents an intermediate physical process fault caused by compromised control. It is not the final hazard, but it can contribute to impact."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-fault-pilot-extinction-due-to-compromised ?node - node)",
        "natural_language_expansion":
            "The pilot flame is extinguished as part of the physical fault-propagation chain.",
        "primary_label": "Impair Process Control",
        "secondary_labels": ["Impact"],
        "justification":
            "Q6: the predicate is an intermediate process consequence before the terminal hazard. It may also indicate Impact because it causes serious process degradation."
    },
    {
        "domain": "Oil-and-gas flare system",
        "pddl_item": "(has-fault-flare-flameout)",
        "natural_language_expansion":
            "The flare experiences a complete flameout, which is the terminal system-wide consequence of the attack.",
        "primary_label": "Impact",
        "secondary_labels": [],
        "justification":
            "Q7: the predicate represents the final physical/operational consequence of the attack."
    },
    {
        "domain": "Water-treatment plant",
        "pddl_item": "(has-caused-chlorine-overfeed-hazard)",
        "natural_language_expansion":
            "A chlorine overfeed hazard has occurred as the terminal consequence of malicious process manipulation.",
        "primary_label": "Impact",
        "secondary_labels": [],
        "justification":
            "Q7: the predicate represents the final hazardous operational impact."
    },
    {
        "domain": "Power-grid substation",
        "pddl_item": "(has-disabled-protection-relay-alarm ?relay - node)",
        "natural_language_expansion":
            "The attacker has disabled a protection relay alarm, preventing the response function from warning operators.",
        "primary_label": "Inhibit Response Function",
        "secondary_labels": ["Impair Process Control"],
        "justification":
            "Q5: the predicate represents disabling an alarm or protection response function. It may also impair process control if the relay affects control behavior."
    },
]

In [36]:
def _format_examples(examples: list[dict]) -> str:
    lines = []

    for i, ex in enumerate(examples, 1):
        body = {
            "pddl_item": ex["pddl_item"],
            "natural_language_expansion": ex["natural_language_expansion"],
            "primary_label": ex["primary_label"],
            "secondary_labels": ex["secondary_labels"],
            "confidence": "high",
            "justification": ex["justification"],
        }

        lines.append(
            f"Example {i} [{ex.get('domain', 'CPS')}]\n"
            f"Input: {ex['pddl_item']}\n"
            + json.dumps(body, indent=2)
        )

    return "\n\n".join(lines)

In [37]:
def build_system_prompt(domain_description: str) -> str:
    """
    Assemble the full MITRE-only multi-label system prompt.
    """

    domain_block = _DOMAIN_CONTEXT_TEMPLATE.format(
        description=domain_description.strip()
    )

    allowed_block = (
        "ALLOWED LABELS\n"
        "==============\n"
        "MITRE ATT&CK for ICS tactics:\n"
        + json.dumps(MITRE_ICS_TACTICS, indent=2)
    )

    example_block = (
        "CROSS-DOMAIN EXAMPLES\n"
        "=====================\n"
        "These examples show how to map PDDL predicates to MITRE ATT&CK for ICS tactics only.\n"
        "Study the reasoning pattern, not only the predicate names.\n\n"
        + _format_examples(_CROSS_DOMAIN_EXAMPLES)
    )

    output_block = (
        "OUTPUT FORMAT\n"
        "=============\n"
        "Return this JSON object and nothing else:\n"
        "{\n"
        '  "pddl_item": "<copy the input verbatim>",\n'
        '  "natural_language_expansion": "<one sentence>",\n'
        '  "primary_label": "<one MITRE ATT&CK for ICS tactic from the allowed list>",\n'
        '  "secondary_labels": ["<zero to three additional MITRE ATT&CK for ICS tactics>"],\n'
        '  "confidence": "<high | medium | low>",\n'
        '  "justification": "<cite the Q-number from the decision tree and explain secondary labels if any>"\n'
        "}"
    )

    return "\n\n".join([
        _RULES,
        domain_block,
        allowed_block,
        example_block,
        output_block,
    ])


SYSTEM_PROMPT = build_system_prompt(CPS_DOMAIN_DESCRIPTION)

In [38]:
# ── 6. Message Builder ───────────────────────────────────────
def build_messages(pddl_item: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"PDDL item: {pddl_item}"},
    ]

In [39]:
# ── 9. Output Parsing and Validation ─────────────────────────

def extract_json(text: str) -> dict | None:
    """
    Extract the first valid JSON object from model output.
    Handles raw JSON and ```json fenced blocks.
    """
    if not isinstance(text, str):
        return None

    text = re.sub(r"```(?:json)?", "", text)
    text = text.replace("```", "").strip()

    # Try direct JSON parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: find JSON-like objects
    candidates = re.findall(r"\{.*?\}", text, re.DOTALL)

    for candidate in reversed(candidates):
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue

    return None


def validate_output(result: dict | None) -> bool:
    """
    Validate that the model returned exactly the fields we need and
    that all labels are from MITRE_ICS_TACTICS.
    """
    if not isinstance(result, dict):
        return False

    required = {
        "pddl_item",
        "natural_language_expansion",
        "primary_label",
        "secondary_labels",
        "confidence",
        "justification",
    }

    if not required.issubset(result.keys()):
        return False

    if result["primary_label"] not in MITRE_ICS_TACTICS:
        return False

    if not isinstance(result["secondary_labels"], list):
        return False

    if len(result["secondary_labels"]) > 3:
        return False

    for label in result["secondary_labels"]:
        if label not in MITRE_ICS_TACTICS:
            return False

        if label == result["primary_label"]:
            return False

    if result["confidence"] not in ["high", "medium", "low"]:
        return False

    return True


def check_against_ground_truth(result: dict) -> str:
    """
    Compare only the primary_label against KNOWN_CORRECT.
    """
    item = result.get("pddl_item", "")

    if item not in KNOWN_CORRECT:
        return "unknown"

    expected_label = KNOWN_CORRECT[item]

    if result.get("primary_label") == expected_label:
        return "correct"

    return f"incorrect -- expected primary: {expected_label}"

In [40]:
# ── 10. Main Mapping Function ────────────────────────────────

def map_pddl_to_mitre_ics(pddl_item: str, retries: int = 2) -> dict:
    messages = build_messages(pddl_item)
    last_raw = ""

    for attempt in range(1, retries + 2):
        try:
            raw_output = generator(messages)[0]["generated_text"]

            # Some chat models return a list of messages
            if isinstance(raw_output, list):
                raw_output = raw_output[-1].get("content", "")

            last_raw = raw_output

            parsed = extract_json(raw_output)

            if validate_output(parsed):
                parsed["attempt"] = attempt
                return parsed

            print(f"    [{pddl_item}] attempt {attempt}: invalid JSON or invalid labels; retrying...")

        except Exception as exc:
            print(f"    [{pddl_item}] attempt {attempt}: {exc}")

    return {
        "pddl_item": pddl_item,
        "natural_language_expansion": None,
        "primary_label": None,
        "secondary_labels": [],
        "confidence": "low",
        "justification": "Model failed to return a valid MITRE ATT&CK for ICS mapping after all attempts.",
        "attempt": retries + 1,
        "raw_output": last_raw,
    }

In [41]:
# ── 11. Run Mapper ───────────────────────────────────────────

print(f"\nDomain : {CPS_DOMAIN_DESCRIPTION[:100].rstrip()} ...")
print(f"Items  : {len(PDDL_ITEMS)}\n")

results = []

for item in PDDL_ITEMS:
    print(f"  -> {item}")

    result = map_pddl_to_mitre_ics(item)
    result["ground_truth_check"] = check_against_ground_truth(result)

    results.append(result)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Domain : An oil-and-gas flare system in which a cyber attacker compromises IT endpoints, pivots through the c ...
Items  : 22

  -> (has-compromised-customer-pc)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  -> (has-compromised-engineering-workstations)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  -> (has-fault-pilot-low-supply-pressure-due-to-compromised ?node - node)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  -> (has-fault-pilot-extinction-due-to-compromised ?node - node)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  -> (has-fault-flare-flameout)


In [42]:
# ── 12. Results Table ────────────────────────────────────────

DISPLAY_COLS = [
    "pddl_item",
    "natural_language_expansion",
    "primary_label",
    "secondary_labels",
    "confidence",
    "justification",
    "ground_truth_check",
    "attempt",
]

df = pd.DataFrame(results)

ordered_cols = [c for c in DISPLAY_COLS if c in df.columns]
extra_cols = [c for c in df.columns if c not in ordered_cols]

df = df[ordered_cols + extra_cols]

pd.set_option("display.max_colwidth", 120)
df

,pddl_item,natural_language_expansion,primary_label,secondary_labels,confidence,justification,ground_truth_check,attempt
0,(has-compromised-customer-pc),"A customer-side PC has already been compromised, giving the attacker an entry point into the environment.",Initial Access,[],high,Q1: the predicate represents a pre-existing foothold used to enter the CPS/ICS environment.,correct,1
1,(has-compromised-engineering-workstations),"Engineering workstations within the CPS/ICS environment have been compromised, providing an initial entry point for ...",Initial Access,[],high,Q1: the predicate represents a pre-existing compromised asset that gives the attacker an entry point into the CPS/IC...,correct,1
2,(has-connected?at - node?to - node),There is a network connection that allows movement from one node to another.,Lateral Movement,[],high,"Q2: the predicate represents network traversal, enabling movement between nodes.",unknown,1
3,(has-access-to-windows-server?node - node),The attacker has access to a Windows Server from another compromised or reachable node.,Lateral Movement,[],high,Q2: the predicate represents movement or reachability to another host.,unknown,1
4,(has-vulnerability-CVE-2019-0575?node - node),"The target node has the CVE-2019-0575 vulnerability, which can be exploited to gain unauthorized access or control.",Execution,[Discovery],high,"Q4: the predicate represents an exploit precondition, indicating a specific vulnerability. It also implies Discovery...",unknown,1
5,(has-vulnerability-CVE-2019-0584?node - node),"The target node has the CVE-2019-0584 vulnerability, which can be exploited to gain unauthorized access or control.",Execution,[Discovery],high,Q4: the predicate represents an exploit precondition. It also implies Discovery because the vulnerable condition has...,unknown,1
6,(has-vulnerability-CVE-2018-0538?node - node),"The target node has the CVE-2018-0538 vulnerability, which can be exploited to gain unauthorized access.",Execution,[Discovery],high,Q4: the predicate represents an exploit precondition. It also implies Discovery because the vulnerable condition has...,unknown,1
7,(has-done-remote-code-execution?node - node),The attacker has successfully executed unauthorized code on the target node.,Execution,[],high,Q4: the predicate represents successful unauthorized code execution.,unknown,1
8,(has-improper-vpn-firewall-configuration?node - node),A misconfiguration of the VPN or firewall allows unauthorized access into a protected network zone.,Initial Access,[Lateral Movement],high,Q1: the predicate represents an exposed entry condition. It also supports Lateral Movement because it enables traver...,unknown,1
9,(has-vulnerability-CVE-2018-0296?node - node),"The target node has the CVE-2018-0296 vulnerability, which can be exploited to gain unauthorized access.",Execution,[Discovery],high,Q4: the predicate represents an exploitable vulnerability. It also implies Discovery because the vulnerable conditio...,unknown,1


In [43]:
# ── 13. Summary ──────────────────────────────────────────────

print("\n-- Primary label distribution -------------------------------")
print(df["primary_label"].value_counts(dropna=False).to_string())

print(f"\nFailed      : {df['primary_label'].isna().sum()} / {len(df)}")
print(f"High conf.  : {(df['confidence'] == 'high').sum()} / {len(df)}")

known_rows = df[df["ground_truth_check"] != "unknown"]

if len(known_rows):
    correct = (known_rows["ground_truth_check"] == "correct").sum()
    incorrect = known_rows[known_rows["ground_truth_check"] != "correct"]

    print(f"GT accuracy : {correct} / {len(known_rows)}")

    if len(incorrect):
        print("\nMismatches vs ground truth:")
        for _, row in incorrect.iterrows():
            print(f"\n{row['pddl_item']}")
            print(f"  Model   : {row['primary_label']}")
            print(f"  Expected: {row['ground_truth_check']}")


-- Primary label distribution -------------------------------
primary_label
Execution                 7
Impair Process Control    6
Lateral Movement          4
Initial Access            3
Discovery                 1
Impact                    1

Failed      : 0 / 22
High conf.  : 22 / 22
GT accuracy : 3 / 3


In [44]:
# ── 14. Export Results ───────────────────────────────────────

OUTPUT_CSV = MAPPING_CSV

df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved -> {OUTPUT_CSV}")

try:
    from google.colab import files
    files.download(OUTPUT_CSV)
except ImportError:
    pass



Saved -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/pddl_mitre_ics_mapping.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Optional manual label corrections

Use this section only when the mapper leaves a predicate unlabeled or assigns a label that the domain expert wants to override.


In [23]:
# # Find the missing row
# missing_label_df = df[df["primary_label"].isna()]
# missing_label_df[["pddl_item", "primary_label"]]


,pddl_item,primary_label


In [24]:
# # Add multiple labels using a dictionary
# manual_labels = {
#     "(has-access-to-windows-server ?node - node)": "Lateral Movement",
#     "(has-fault-manual-isolation-valve-close-due-to-compromised ?node - node)": "Trustworthiness - Safety",
# }

# for pddl_item, label in manual_labels.items():
#     df.loc[df["pddl_item"] == pddl_item, "selected_label"] = label


In [25]:
# Check for NaN again
# df[df["selected_label"].isna()][["pddl_item", "selected_label"]]


,pddl_item,selected_label
0,(has-compromised-customer-pc),NaN
1,(has-compromised-engineering-workstations),NaN
2,(has-connected?at - node?to - node),NaN
3,(has-access-to-windows-server?node - node),NaN
4,(has-vulnerability-CVE-2019-0575?node - node),NaN
5,(has-vulnerability-CVE-2019-0584?node - node),NaN
6,(has-vulnerability-CVE-2018-0538?node - node),NaN
7,(has-done-remote-code-execution?node - node),NaN
8,(has-improper-vpn-firewall-configuration?node - node),NaN
9,(has-vulnerability-CVE-2018-0296?node - node),NaN


In [ ]:
# df.to_csv(MAPPING_CSV, index=False)
# print(f"Updated file saved -> {MAPPING_CSV}")


## 5. Build FCA context and abstraction nodes


In [45]:
import pandas as pd
from itertools import combinations

# df is the output from your mapper.
# Required columns:
# pddl_item, selected_framework, primary_label

def build_fca_context_from_labels(df):
    object_attributes = {}

    for _, row in df.iterrows():
        obj = row["pddl_item"]

        attrs = {
            str(row["primary_label"])
        }

        object_attributes[obj] = attrs

    all_attributes = sorted(set().union(*object_attributes.values()))

    rows = []
    for obj, attrs in object_attributes.items():
        row = {"pddl_item": obj}
        for attr in all_attributes:
            row[attr] = 1 if attr in attrs else 0
        rows.append(row)

    context_df = pd.DataFrame(rows)
    return context_df, object_attributes, all_attributes


context_df, object_attributes, all_attributes = build_fca_context_from_labels(df)
context_df


,pddl_item,Discovery,Execution,Impact,Impair Process Control,Initial Access,Lateral Movement
0,(has-compromised-customer-pc),0,0,0,0,1,0
1,(has-compromised-engineering-workstations),0,0,0,0,1,0
2,(has-connected?at - node?to - node),0,0,0,0,0,1
3,(has-access-to-windows-server?node - node),0,0,0,0,0,1
4,(has-vulnerability-CVE-2019-0575?node - node),0,1,0,0,0,0
5,(has-vulnerability-CVE-2019-0584?node - node),0,1,0,0,0,0
6,(has-vulnerability-CVE-2018-0538?node - node),0,1,0,0,0,0
7,(has-done-remote-code-execution?node - node),0,1,0,0,0,0
8,(has-improper-vpn-firewall-configuration?node - node),0,0,0,0,1,0
9,(has-vulnerability-CVE-2018-0296?node - node),0,1,0,0,0,0


In [46]:
context_df = context_df.rename(columns={
    "Initial Access" : "C1",
    "Execution" : "C2",
    "Persistence" : "C3",
    "Privilege Escalation": "C4",
    "Evasion" : "C9",
    "Discovery" : "C6",
    "Lateral Movement" : "C7",
    "Collection" : "C8",
    "Command and Control" : "C5",
    "Inhibit Response Function" : "C10",
    "Impair Process Control" : "C11",
    "Impact" : "C12"
})


In [47]:
MIT0_ALLOWED = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C12"]
MOT0_ALLOWED = ["C2", "C7", "C10", "C11", "C12"]


In [48]:
def create_node_from_available_concepts(df, allowed_concepts, node_name, base_cols=["pddl_item"]):
    """
    Creates a node dataframe using only the allowed concept columns
    that are actually available in df.
    """

    # Concepts that exist as columns in df
    available_concepts = [c for c in allowed_concepts if c in df.columns]

    # Concepts allowed but missing from df
    missing_concepts = [c for c in allowed_concepts if c not in df.columns]

    # Keep base columns if they exist, e.g., pddl_item
    existing_base_cols = [c for c in base_cols if c in df.columns]

    # Final columns for this abstraction node
    selected_cols = existing_base_cols + available_concepts

    node_df = df[selected_cols].copy()

    print(f"{node_name} available concepts:", available_concepts)
    print(f"{node_name} missing concepts:", missing_concepts)
    print(f"{node_name} dataframe shape:", node_df.shape)

    return node_df


MIT0 = create_node_from_available_concepts(
    context_df,
    MIT0_ALLOWED,
    "MIT0"
)

MOT0 = create_node_from_available_concepts(
    context_df,
    MOT0_ALLOWED,
    "MOT0"
)


MIT0 available concepts: ['C1', 'C2', 'C6', 'C7', 'C12']
MIT0 missing concepts: ['C3', 'C4', 'C5', 'C8', 'C9']
MIT0 dataframe shape: (22, 6)
MOT0 available concepts: ['C2', 'C7', 'C11', 'C12']
MOT0 missing concepts: ['C10']
MOT0 dataframe shape: (22, 5)


In [49]:
OUTPUT_CSV = BASE_DIR + "/MOT0.csv"
MOT0.to_csv(OUTPUT_CSV, index=False)
print(f"MOT0 file saved -> {OUTPUT_CSV}")

OUTPUT_CSV = BASE_DIR + "/MIT0.csv"
MIT0.to_csv(OUTPUT_CSV, index=False)
print(f"MIT0 file saved -> {OUTPUT_CSV}")


MOT0 file saved -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/MOT0.csv
MIT0 file saved -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/MIT0.csv


In [50]:
def get_concepts_from_node_df(node_df, base_cols=["pddl_item"]):
    """
    Extract concept columns from an existing node dataframe.
    Excludes base columns such as pddl_item.
    """
    return [c for c in node_df.columns if c not in base_cols]


MIT0_concepts = get_concepts_from_node_df(MIT0)
MOT0_concepts = get_concepts_from_node_df(MOT0)

print("MIT0 concepts:", MIT0_concepts)
print("MOT0 concepts:", MOT0_concepts)


MIT0 concepts: ['C1', 'C2', 'C6', 'C7', 'C12']
MOT0 concepts: ['C2', 'C7', 'C11', 'C12']


In [51]:
# Concepts visible at the most detailed node for MOT
def create_gradual_context_nodes_MOT(
    context_df,
    concepts,
    prefix="MOT",
    base_cols=["pddl_item"],
    always_keep=["C12"]
):
    """
    Create abstraction-node dataframes by gradually removing concept columns
    from the front/ascending order, while always keeping C12 fixed.

    Example:
      concepts = ["C2", "C7", "C10", "C11", "C12"]
      always_keep = ["C12"]

      MOT0: pddl_item + C2 + C7 + C10 + C11 + C12
      MOT1: pddl_item + C7 + C10 + C11 + C12
      MOT2: pddl_item + C10 + C11 + C12
      MOT3: pddl_item + C11 + C12
      MOT4: pddl_item + C12
    """

    node_dfs = {}
    node_concepts = {}
    node_summary = []

    # Concepts that can be removed gradually
    removable_concepts = [
        c for c in concepts
        if c not in always_keep
    ]

    # Concepts that must always stay
    fixed_concepts = [
        c for c in always_keep
        if c in concepts
    ]

    for i in range(len(removable_concepts)):
        node_name = f"{prefix}{i}"

        # Remove from the front/ascending order
        retained_removable = removable_concepts[i:]

        # Always keep C12 fixed
        retained_concepts = retained_removable + fixed_concepts

        # Keep only concepts that actually exist as columns in context_df
        available_concepts = [
            c for c in retained_concepts
            if c in context_df.columns
        ]

        missing_concepts = [
            c for c in retained_concepts
            if c not in context_df.columns
        ]

        # Keep base columns such as pddl_item if they exist
        available_base_cols = [
            c for c in base_cols
            if c in context_df.columns
        ]

        selected_cols = available_base_cols + available_concepts

        # Important: keep all rows, only reduce columns
        node_df = context_df[selected_cols].copy()

        node_dfs[node_name] = node_df
        node_concepts[node_name] = available_concepts

        node_summary.append({
            "node": node_name,
            "retained_concepts_requested": retained_concepts,
            "retained_concepts_available": available_concepts,
            "missing_concepts": missing_concepts,
            "num_rows": node_df.shape[0],
            "num_columns": node_df.shape[1],
        })

    node_summary_df = pd.DataFrame(node_summary)

    return node_dfs, node_concepts, node_summary_df


In [52]:
def create_gradual_context_nodes_MIT(
    context_df,
    concepts,
    prefix="MIT",
    base_cols=["pddl_item"],
    always_keep=["C12"]
):
    """
    Create abstraction-node dataframes by gradually removing concept columns
    from the end, while always keeping selected concepts such as C12.

    Example:
      concepts = ["C2", "C7", "C10", "C11", "C12"]
      always_keep = ["C12"]

      MOT0: pddl_item + C2 + C7 + C10 + C11 + C12
      MOT1: pddl_item + C2 + C7 + C10 + C12
      MOT2: pddl_item + C2 + C7 + C12
      MOT3: pddl_item + C2 + C12
      MOT4: pddl_item + C12
    """

    node_dfs = {}
    node_concepts = {}
    node_summary = []

    # Concepts that can be removed
    removable_concepts = [c for c in concepts if c not in always_keep]

    for i in range(len(removable_concepts)):
        node_name = f"{prefix}{i}"

        # Remove from the end of removable concepts
        retained_removable = removable_concepts[:len(removable_concepts) - i]

        # Always keep C12 or any concept listed in always_keep
        retained_concepts = retained_removable + [
            c for c in always_keep if c in concepts
        ]

        # Keep only concepts that actually exist as columns in context_df
        available_concepts = [
            c for c in retained_concepts
            if c in context_df.columns
        ]

        missing_concepts = [
            c for c in retained_concepts
            if c not in context_df.columns
        ]

        available_base_cols = [
            c for c in base_cols
            if c in context_df.columns
        ]

        selected_cols = available_base_cols + available_concepts

        node_df = context_df[selected_cols].copy()

        node_dfs[node_name] = node_df
        node_concepts[node_name] = available_concepts

        node_summary.append({
            "node": node_name,
            "retained_concepts_requested": retained_concepts,
            "retained_concepts_available": available_concepts,
            "missing_concepts": missing_concepts,
            "num_rows": node_df.shape[0],
            "num_columns": node_df.shape[1],
        })

    node_summary_df = pd.DataFrame(node_summary)

    return node_dfs, node_concepts, node_summary_df


In [53]:
MIT_node_dfs, MIT_node_concepts, MIT_node_summary = create_gradual_context_nodes_MIT(
    context_df=context_df,
    concepts=MIT0_concepts,
    prefix="MIT",
    always_keep=["C12"]
)

MIT_node_summary


,node,retained_concepts_requested,retained_concepts_available,missing_concepts,num_rows,num_columns
0,MIT0,"[C1, C2, C6, C7, C12]","[C1, C2, C6, C7, C12]",[],22,6
1,MIT1,"[C1, C2, C6, C12]","[C1, C2, C6, C12]",[],22,5
2,MIT2,"[C1, C2, C12]","[C1, C2, C12]",[],22,4
3,MIT3,"[C1, C12]","[C1, C12]",[],22,3


In [54]:
MOT_node_dfs, MOT_node_concepts, MOT_node_summary = create_gradual_context_nodes_MOT(
    context_df=context_df,
    concepts=MOT0_concepts,
    prefix="MOT",
    always_keep=["C12"]
)

MOT_node_summary


,node,retained_concepts_requested,retained_concepts_available,missing_concepts,num_rows,num_columns
0,MOT0,"[C2, C7, C11, C12]","[C2, C7, C11, C12]",[],22,5
1,MOT1,"[C7, C11, C12]","[C7, C11, C12]",[],22,4
2,MOT2,"[C11, C12]","[C11, C12]",[],22,3


In [55]:
def filter_rows_with_any_one(node_df, base_cols=["pddl_item"]):
    """
    Keeps only rows where at least one concept column has value 1.
    Works even if the values are strings like "1".
    """

    concept_cols = [c for c in node_df.columns if c not in base_cols]

    if not concept_cols:
        return node_df.copy()

    numeric_part = (
        node_df[concept_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
    )

    filtered_df = node_df[numeric_part.eq(1).any(axis=1)].copy()

    return filtered_df


# Filter MIT nodes
MIT_node_dfs_filtered = {
    node_name: filter_rows_with_any_one(node_df)
    for node_name, node_df in MIT_node_dfs.items()
}

# Filter MOT nodes
MOT_node_dfs_filtered = {
    node_name: filter_rows_with_any_one(node_df)
    for node_name, node_df in MOT_node_dfs.items()
}


In [56]:
for node_name, node_df in MIT_node_dfs.items():
    filtered_df = MIT_node_dfs_filtered[node_name]
    print(f"{node_name}: before={node_df.shape}, after={filtered_df.shape}")

for node_name, node_df in MOT_node_dfs.items():
    filtered_df = MOT_node_dfs_filtered[node_name]
    print(f"{node_name}: before={node_df.shape}, after={filtered_df.shape}")


MIT0: before=(22, 6), after=(16, 6)
MIT1: before=(22, 5), after=(12, 5)
MIT2: before=(22, 4), after=(11, 4)
MIT3: before=(22, 3), after=(4, 3)
MOT0: before=(22, 5), after=(18, 5)
MOT1: before=(22, 4), after=(11, 4)
MOT2: before=(22, 3), after=(7, 3)


In [57]:
# Store each filtered node as CSV
from pathlib import Path

OUTPUT_DIR = Path(FILTERED_NODE_DIR)
OUTPUT_DIR.mkdir(exist_ok=True)

for node_name, node_df in MIT_node_dfs_filtered.items():
    output_path = OUTPUT_DIR / f"{node_name}.csv"
    node_df.to_csv(output_path, index=False)
    print(f"Saved {node_name} -> {output_path}")

for node_name, node_df in MOT_node_dfs_filtered.items():
    output_path = OUTPUT_DIR / f"{node_name}.csv"
    node_df.to_csv(output_path, index=False)
    print(f"Saved {node_name} -> {output_path}")


Saved MIT0 -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MIT0.csv
Saved MIT1 -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MIT1.csv
Saved MIT2 -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MIT2.csv
Saved MIT3 -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MIT3.csv
Saved MOT0 -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MOT0.csv
Saved MOT1 -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MOT1.csv
Saved MOT2 -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MOT2.csv


In [58]:
# MOST_ABSTRACTED_NODE = "M_ABS"
MOST_ABSTRACTED_CONCEPTS = ["C12"]

FULL_LABEL = "Full Ground Truth Model"

def remove_abstract_duplicate_nodes(node_concepts, abstract_concepts):
    """
    Removes nodes that duplicate the most abstract node.

    Example:
      MIT4 = C12 should be removed because M_ABS already represents C12.
      MOT3 = C12 should be removed because M_ABS already represents C12.
    """
    abstract_set = set(abstract_concepts)
    cleaned = {}

    for node_name, concepts in node_concepts.items():
        if set(concepts) == abstract_set:
            print(f"Removing {node_name} because it duplicates {MOST_ABSTRACTED_NODE}")
            continue

        cleaned[node_name] = concepts

    return cleaned


MIT_node_concepts = remove_abstract_duplicate_nodes(
    MIT_node_concepts,
    MOST_ABSTRACTED_CONCEPTS
)

MOT_node_concepts = remove_abstract_duplicate_nodes(
    MOT_node_concepts,
    MOST_ABSTRACTED_CONCEPTS
)

print("MIT nodes kept:", list(MIT_node_concepts.keys()))
print("MOT nodes kept:", list(MOT_node_concepts.keys()))


MIT nodes kept: ['MIT0', 'MIT1', 'MIT2', 'MIT3']
MOT nodes kept: ['MOT0', 'MOT1', 'MOT2']


## 6. Build concept-to-predicate mappings for bridge-node metadata


In [59]:
import pandas as pd
import re

def clean_pddl_item(item):
    """
    Converts:
    (has-connected?at - node?to - node)
    into:
    has-connected
    """
    item = str(item).strip()

    # remove surrounding parentheses
    item = item.strip("()")

    # keep only predicate name before parameters
    item = re.split(r"\?", item)[0]

    return item.strip()


def concept_sort_key(concept):
    """
    Sorts concepts like C1, C2, C6, C7, C11, C12 numerically.
    """
    match = re.search(r"\d+", concept)
    return int(match.group()) if match else 999


def build_concept_mapping(df):
    concept_cols = [col for col in df.columns if col != "pddl_item"]
    concept_cols = sorted(concept_cols, key=concept_sort_key)

    concept_mapping = {}

    for concept in concept_cols:
        items = df.loc[df[concept] == 1, "pddl_item"].tolist()
        cleaned_items = [clean_pddl_item(item) for item in items]

        # remove duplicates while preserving order
        cleaned_items = list(dict.fromkeys(cleaned_items))

        concept_mapping[concept] = cleaned_items

    return concept_mapping


def print_concept_mapping(concept_mapping, variable_name="concept_mapping"):
    print(f"{variable_name} = {{")

    concepts = list(concept_mapping.keys())

    for i, concept in enumerate(concepts):
        items = concept_mapping[concept]

        if len(items) == 0:
            line = f'    "{concept}" : []'
            if i < len(concepts) - 1:
                line += ","
            print(line)
            continue

        print(f'    "{concept}" : [', end="")

        for j, item in enumerate(items):
            if j == 0:
                print(f'"{item}"')
            else:
                print(f'            "{item}"')

            if j < len(items) - 1:
                print("            ,", end="")

        print("            ]", end="")

        if i < len(concepts) - 1:
            print(",")
        else:
            print()

    print("}")

In [60]:
MIT_concept_mapping = build_concept_mapping(MIT0)
MOT_concept_mapping = build_concept_mapping(MOT0)

#Uncomment these lines if you want to inspect the mappings.
print_concept_mapping(MIT_concept_mapping, "MIT_concept_mapping")
print_concept_mapping(MOT_concept_mapping, "MOT_concept_mapping")


MIT_concept_mapping = {
    "C1" : ["has-compromised-customer-pc"
            ,            "has-compromised-engineering-workstations"
            ,            "has-improper-vpn-firewall-configuration"
            ],
    "C2" : ["has-vulnerability-CVE-2019-0575"
            ,            "has-vulnerability-CVE-2019-0584"
            ,            "has-vulnerability-CVE-2018-0538"
            ,            "has-done-remote-code-execution"
            ,            "has-vulnerability-CVE-2018-0296"
            ,            "has-vulnerability-CVE-2017-9312"
            ,            "has-exploit-improperly-implemented-fields-in-stack"
            ],
    "C6" : ["has-Allen-Bradley-L30ERMS-safety-devices-v30-and-earlier"
            ],
    "C7" : ["has-connected"
            ,            "has-access-to-windows-server"
            ,            "has-access-to-network"
            ,            "has-access-to-plc-port-1132-tcp"
            ],
    "C12" : ["has-fault-flare-flameout"
            ]
}
MO

In [61]:
from pathlib import Path
import pandas as pd
import re

In [62]:
# ============================================================
# Reference FULL files
# ============================================================

FULL_DOMAIN_FILE = Path(DOMAIN_PDDL)
FULL_PROBLEM_FILE = Path(PROBLEM_PDDL)

FILTERED_NODE_DIR_PATH = Path(FILTERED_NODE_DIR)
OUTPUT_BASE_DIR_PATH = Path(OUTPUT_BASE_DIR)

print("Reading FULL reference files from:")
print(FULL_DOMAIN_FILE)
print(FULL_PROBLEM_FILE)

print("\nReading node CSV files from:")
print(FILTERED_NODE_DIR_PATH)

print("\nWriting abstracted models to:")
print(OUTPUT_BASE_DIR_PATH)


Reading FULL reference files from:
/content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/FULL/domain.pddl
/content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/FULL/problem.pddl

Reading node CSV files from:
/content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv

Writing abstracted models to:
/content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model


In [63]:
# ============================================================
# Helper functions
# ============================================================

def read_file(path):
    return Path(path).read_text(encoding="utf-8")


def write_file(path, text):
    Path(path).write_text(text, encoding="utf-8")


def clean_predicate_name(item):
    item = str(item).strip().strip("()")
    item = re.split(r"\?", item)[0]
    item = item.split()[0]
    return item.strip()


def parse_pddl_list(text):
    tokens = re.findall(r"\(|\)|[^\s()]+", text)

    def parse_expr(index):
        result = []
        while index < len(tokens):
            tok = tokens[index]
            if tok == "(":
                sub, index = parse_expr(index + 1)
                result.append(sub)
            elif tok == ")":
                return result, index + 1
            else:
                result.append(tok)
                index += 1
        return result, index

    parsed, _ = parse_expr(0)
    return parsed[0]


def flatten(expr):
    result = []
    for x in expr:
        if isinstance(x, list):
            result.extend(flatten(x))
        else:
            result.append(x)
    return result


def predicate_name(atom):
    if isinstance(atom, list) and atom:
        return atom[0]
    return None


def filter_condition(expr, allowed_predicates):
    if not isinstance(expr, list):
        return expr

    if not expr:
        return expr

    if expr[0] == "and":
        kept = []
        for item in expr[1:]:
            filtered = filter_condition(item, allowed_predicates)
            if filtered is not None:
                kept.append(filtered)
        return ["and"] + kept

    pred = predicate_name(expr)

    if pred in allowed_predicates:
        return expr

    return None


def to_pddl(expr, indent=0):
    space = " " * indent

    if isinstance(expr, str):
        return expr

    if not expr:
        return "()"

    head = expr[0]

    if head == "define":
        lines = ["(define"]
        for item in expr[1:]:
            lines.append(to_pddl(item, 2))
        lines.append(")")
        return "\n".join(lines)

    if head == ":requirements":
        return f"{space}(:requirements {' '.join(expr[1:])})"

    if head == ":types":
        return f"{space}(:types {' '.join(expr[1:])})"

    if head == ":domain":
        return f"{space}(:domain {' '.join(expr[1:])})"

    if head == ":objects":
        return f"{space}(:objects {' '.join(flatten(expr[1:]))})"

    if head == ":predicates":
        lines = [f"{space}(:predicates"]
        for item in expr[1:]:
            lines.append(to_pddl(item, indent + 2))
        lines.append(f"{space})")
        return "\n".join(lines)

    if head == ":init":
        lines = [f"{space}(:init"]
        for item in expr[1:]:
            lines.append(to_pddl(item, indent + 2))
        lines.append(f"{space})")
        return "\n".join(lines)

    if head == ":goal":
        return f"{space}(:goal\n{to_pddl(expr[1], indent + 2)}\n{space})"

    if head == ":action":
        lines = [f"{space}(:action {expr[1]}"]
        i = 2
        while i < len(expr):
            key = expr[i]
            val = expr[i + 1]

            if key == ":parameters":
                params = " ".join(flatten(val))
                lines.append(f"{space}  :parameters ({params})")
            elif key in [":precondition", ":effect"]:
                lines.append(f"{space}  {key}")
                lines.append(to_pddl(val, indent + 4))
            else:
                lines.append(f"{space}  {key} {to_pddl(val)}")

            i += 2

        lines.append(f"{space})")
        return "\n".join(lines)

    if head == "and":
        if len(expr) == 1:
            return f"{space}(and)"
        lines = [f"{space}(and"]
        for item in expr[1:]:
            lines.append(to_pddl(item, indent + 2))
        lines.append(f"{space})")
        return "\n".join(lines)

    return f"{space}(" + " ".join(to_pddl(x).strip() for x in expr) + ")"

In [64]:
# ============================================================
# Update domain
# ============================================================

def update_domain(domain_text, allowed_predicates):
    parsed = parse_pddl_list(domain_text)
    updated = []

    for section in parsed:
        if isinstance(section, list) and section:
            if section[0] == ":predicates":
                kept_preds = [
                    pred for pred in section[1:]
                    if predicate_name(pred) in allowed_predicates
                ]
                updated.append([":predicates"] + kept_preds)

            elif section[0] == ":action":
                new_action = section.copy()

                i = 2
                while i < len(new_action):
                    key = new_action[i]

                    if key in [":precondition", ":effect"]:
                        filtered = filter_condition(new_action[i + 1], allowed_predicates)

                        if filtered is None:
                            filtered = ["and"]

                        new_action[i + 1] = filtered

                    i += 2

                updated.append(new_action)

            else:
                updated.append(section)
        else:
            updated.append(section)

    return to_pddl(updated)


In [65]:
# ============================================================
# Update problem
# ============================================================

def update_problem(problem_text, allowed_predicates):
    parsed = parse_pddl_list(problem_text)
    updated = []

    for section in parsed:
        if isinstance(section, list) and section:
            if section[0] == ":init":
                kept_init = [
                    atom for atom in section[1:]
                    if predicate_name(atom) in allowed_predicates
                ]
                updated.append([":init"] + kept_init)

            elif section[0] == ":goal":
                filtered_goal = filter_condition(section[1], allowed_predicates)

                if filtered_goal is None:
                    filtered_goal = ["and"]

                updated.append([":goal", filtered_goal])

            else:
                updated.append(section)
        else:
            updated.append(section)

    return to_pddl(updated)

In [66]:
# ============================================================
# Generate one domain/problem pair for each CSV node
# ============================================================

full_domain_text = read_file(FULL_DOMAIN_FILE)
full_problem_text = read_file(FULL_PROBLEM_FILE)

csv_files = sorted(FILTERED_NODE_DIR_PATH.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {FILTERED_NODE_DIR_PATH}")

for CSV_FILE in csv_files:
    NODE_NAME = CSV_FILE.stem
    OUTPUT_DIR = OUTPUT_BASE_DIR_PATH / NODE_NAME

    OUTPUT_DOMAIN_FILE = OUTPUT_DIR / "domain.pddl"
    OUTPUT_PROBLEM_FILE = OUTPUT_DIR / "problem.pddl"

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(CSV_FILE)

    if "pddl_item" not in df.columns:
        raise ValueError(f"{CSV_FILE} must contain a column named 'pddl_item'.")

    allowed_predicates = set(df["pddl_item"].apply(clean_predicate_name))

    updated_domain = update_domain(full_domain_text, allowed_predicates)
    updated_problem = update_problem(full_problem_text, allowed_predicates)

    write_file(OUTPUT_DOMAIN_FILE, updated_domain)
    write_file(OUTPUT_PROBLEM_FILE, updated_problem)

    print("=" * 70)
    print(f"Created node: {NODE_NAME}")
    print(f"CSV: {CSV_FILE}")
    print(f"Domain: {OUTPUT_DOMAIN_FILE}")
    print(f"Problem: {OUTPUT_PROBLEM_FILE}")
    print(f"Predicates retained: {len(allowed_predicates)}")

Created node: MIT0
CSV: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MIT0.csv
Domain: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT0/domain.pddl
Problem: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT0/problem.pddl
Predicates retained: 16
Created node: MIT1
CSV: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MIT1.csv
Domain: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT1/domain.pddl
Problem: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT1/problem.pddl
Predicates retained: 12
Created node: MIT2
CSV: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/filtered_node_csv/MIT2.csv
Domain: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT2/domain.pddl
Problem: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT2/problem.

**Removed redundant cell.** Predicate loading is handled inside the node-generation loop above.

## 7. Create final lattice with bridge nodes


In [67]:
import json
import re
from pathlib import Path
from itertools import product

# ============================================================
# Configuration
# ============================================================

OUTPUT_DIR = Path(LATTICE_OUTPUT_DIR)
OUTPUT_DIR.mkdir(exist_ok=True)

LATTICE_JSON_PATH = OUTPUT_DIR / "eval_lattice_with_bridge_nodes.json"

DOMAIN_FILE = "domain.pddl"
PROBLEM_FILE = "problem.pddl"

MOST_ABSTRACTED_NODE = "M_ABS"
MOST_ABSTRACTED_CONCEPTS = ["C12"]

FULL_NODE = "FULL"


In [69]:
print(LATTICE_JSON_PATH)

/content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/lattice_output/eval_lattice_with_bridge_nodes.json


In [70]:
# ============================================================
# Helpers
# ============================================================

def concept_sort_key(concept):
    """
    Sorts C1, C2, C7, C11, C12 or c1, c2, c7, c11, c12 numerically.
    """
    match = re.search(r"\d+", str(concept))
    return int(match.group()) if match else 999


def to_high_level_concept(low_level_concept):
    """
    Convert low-level concept C11 into high-level concept c11.
    """
    return low_level_concept.lower()


def make_model_paths(node_name):
    """
    Store model paths relative to the configured OUTPUT_BASE_DIR.

    Example:
      OUTPUT_BASE_DIR = ".../model"
      node_name = "MIT3-MOT1"

    JSON path:
      model/MIT3-MOT1/domain.pddl
      model/MIT3-MOT1/problem.pddl
    """
    model_dir_name = Path(OUTPUT_BASE_DIR).name

    return {
        "domain": f"{model_dir_name}/{node_name}/{DOMAIN_FILE}",
        "problem": f"{model_dir_name}/{node_name}/{PROBLEM_FILE}",
    }


def sorted_concepts(concepts):
    return sorted(list(concepts), key=concept_sort_key)


def format_label(low_level_concepts=None, high_level_concepts=None):
    """
    Creates readable labels such as:
        C1, C2, C12; c7, c11
    """
    low_level_concepts = low_level_concepts or []
    high_level_concepts = high_level_concepts or []

    parts = []

    if low_level_concepts:
        parts.append(", ".join(sorted_concepts(low_level_concepts)))

    if high_level_concepts:
        parts.append(", ".join(sorted_concepts(high_level_concepts)))

    return "; ".join(parts)


In [71]:
# ============================================================
# MITRE ATT&CK for ICS concept metadata
# ============================================================

CONCEPT_TO_MITRE_ICS_TACTIC = {
    "C1": "Initial Access",
    "C2": "Execution",
    "C3": "Persistence",
    "C4": "Privilege Escalation",
    "C5": "Command and Control",
    "C6": "Discovery",
    "C7": "Lateral Movement",
    "C8": "Collection",
    "C9": "Evasion",
    "C10": "Inhibit Response Function",
    "C11": "Impair Process Control",
    "C12": "Impact",
}

MITRE_ICS_TACTIC_MEANINGS = {
    "Initial Access": "The adversary is trying to get into the ICS environment.",
    "Execution": "The adversary is trying to run code or manipulate system functions, parameters, and data in an unauthorized way.",
    "Persistence": "The adversary is trying to maintain their foothold in the ICS environment.",
    "Privilege Escalation": "The adversary is trying to gain higher-level permissions.",
    "Evasion": "The adversary is trying to avoid security defenses.",
    "Discovery": "The adversary is locating information to assess and identify targets in the ICS environment.",
    "Lateral Movement": "The adversary is trying to move through the ICS environment.",
    "Collection": "The adversary is trying to gather data and domain knowledge from the ICS environment.",
    "Command and Control": "The adversary is trying to communicate with and control compromised systems.",
    "Inhibit Response Function": "The adversary is trying to prevent safety, protection, or response functions from operating correctly.",
    "Impair Process Control": "The adversary is trying to disrupt, degrade, or manipulate control of the physical process.",
    "Impact": "The adversary is trying to cause a harmful operational, safety, or physical consequence.",
}


def concept_metadata(low_level_concept):
    tactic = CONCEPT_TO_MITRE_ICS_TACTIC.get(low_level_concept, "Unmapped")
    return {
        "low_level_concept": low_level_concept,
        "high_level_concept": to_high_level_concept(low_level_concept),
        "mitre_attack_ics_tactic": tactic,
        "meaning": MITRE_ICS_TACTIC_MEANINGS.get(
            tactic,
            "No MITRE ATT&CK for ICS meaning defined."
        )
    }


In [72]:
# ============================================================
# Build base lattice nodes
# Uses MIT_node_concepts and MOT_node_concepts from your notebook
# ============================================================

nodes = {}

# Total low-level concept universe
FULL_CONCEPTS = sorted_concepts(
    set().union(
        *MIT_node_concepts.values(),
        *MOT_node_concepts.values(),
        MOST_ABSTRACTED_CONCEPTS
    )
)

ALL_HIGH_LEVEL_CONCEPTS = [
    to_high_level_concept(c)
    for c in FULL_CONCEPTS
]

# Most abstract node
nodes[MOST_ABSTRACTED_NODE] = {
    **make_model_paths(MOST_ABSTRACTED_NODE),
    "node_type": "abstract",
    "label": format_label(low_level_concepts=MOST_ABSTRACTED_CONCEPTS),
    "retained_low_level_concepts": sorted_concepts(MOST_ABSTRACTED_CONCEPTS),
    "retained_high_level_concepts": [],
    "missing_low_level_concepts_from_full": sorted_concepts(
        set(FULL_CONCEPTS) - set(MOST_ABSTRACTED_CONCEPTS)
    ),
}

# Full node
nodes[FULL_NODE] = {
    **make_model_paths(FULL_NODE),
    "node_type": "full",
    "label": "Full Ground Truth Model",
    "retained_low_level_concepts": sorted_concepts(FULL_CONCEPTS),
    "retained_high_level_concepts": [],
    "missing_low_level_concepts_from_full": [],
}

# MIT branch nodes
for node_name, concepts in MIT_node_concepts.items():
    retained = sorted_concepts(concepts)
    missing = sorted_concepts(set(FULL_CONCEPTS) - set(retained))

    nodes[node_name] = {
        **make_model_paths(node_name),
        "node_type": "MIT",
        "label": format_label(low_level_concepts=retained),
        "retained_low_level_concepts": retained,
        "retained_high_level_concepts": [],
        "missing_low_level_concepts_from_full": missing,
    }

# MOT branch nodes
for node_name, concepts in MOT_node_concepts.items():
    retained = sorted_concepts(concepts)
    missing = sorted_concepts(set(FULL_CONCEPTS) - set(retained))

    nodes[node_name] = {
        **make_model_paths(node_name),
        "node_type": "MOT",
        "label": format_label(low_level_concepts=retained),
        "retained_low_level_concepts": retained,
        "retained_high_level_concepts": [],
        "missing_low_level_concepts_from_full": missing,
    }


In [73]:
# ============================================================
# Main lattice refinement edges
# Direction: more abstract -> more detailed
# ============================================================

edges = []


def added_low_level_concepts(from_node, to_node):
    from_concepts = set(nodes[from_node]["retained_low_level_concepts"])
    to_concepts = set(nodes[to_node]["retained_low_level_concepts"])
    return sorted_concepts(to_concepts - from_concepts)


def make_refinement_edge(from_node, to_node):
    added = added_low_level_concepts(from_node, to_node)

    return {
        "from": from_node,
        "to": to_node,
        "edge_type": "refinement",
        "added_low_level_concepts": added,
        "label": "add " + ", ".join(added) if added else "same abstraction",
    }


# MIT branch order: M_ABS -> most abstract MIT -> ... -> most detailed MIT -> FULL
MIT_ORDER = [MOST_ABSTRACTED_NODE] + sorted(
    MIT_node_concepts.keys(),
    key=lambda n: int(re.search(r"\d+", n).group()),
    reverse=True
) + [FULL_NODE]

for src, dst in zip(MIT_ORDER[:-1], MIT_ORDER[1:]):
    edges.append(make_refinement_edge(src, dst))


# MOT branch order: M_ABS -> most abstract MOT -> ... -> most detailed MOT -> FULL
MOT_ORDER = [MOST_ABSTRACTED_NODE] + sorted(
    MOT_node_concepts.keys(),
    key=lambda n: int(re.search(r"\d+", n).group()),
    reverse=True
) + [FULL_NODE]

for src, dst in zip(MOT_ORDER[:-1], MOT_ORDER[1:]):
    edges.append(make_refinement_edge(src, dst))


In [74]:
def node_sort_key(node_name):
    match = re.search(r"\d+", str(node_name))
    return int(match.group()) if match else 999


In [75]:
# ============================================================
# Bridge node creation
# ============================================================

def create_bridge_node(source_node, target_node):
    """
    Creates one bridge node.

    Rule:
      bridge retained low-level concepts = source retained low-level concepts
      bridge retained high-level concepts = high-level version of
          target concepts missing from source

    Example:
      source MIT3 = C1, C12
      target MOT0 = C2, C7, C11, C12

      missing = C2, C7, C11
      bridge = C1, C12 + c2, c7, c11
    """

    source_low = set(nodes[source_node]["retained_low_level_concepts"])
    target_low = set(nodes[target_node]["retained_low_level_concepts"])

    missing_from_source = sorted_concepts(target_low - source_low)
    added_high = sorted_concepts([
        to_high_level_concept(c)
        for c in missing_from_source
    ])

    bridge_name = f"{source_node}-{target_node}"

    high_level_metadata = {
        to_high_level_concept(c): concept_metadata(c)
        for c in missing_from_source
    }

    bridge_node = {
        **make_model_paths(bridge_name),

        "node_type": "bridge",
        "source_node": source_node,
        "target_node": target_node,

        "label": format_label(
            low_level_concepts=sorted_concepts(source_low),
            high_level_concepts=added_high
        ),

        "retained_low_level_concepts": sorted_concepts(source_low),
        "retained_high_level_concepts": added_high,

        "source_low_level_concepts": sorted_concepts(source_low),
        "target_low_level_concepts": sorted_concepts(target_low),

        "missing_low_level_concepts_from_source": missing_from_source,
        "added_high_level_concepts": added_high,

        "high_level_concept_meanings": high_level_metadata,

        "explanation_rule": (
            "This bridge node keeps the source node's low-level concepts and "
            "adds high-level concepts for the target node's low-level concepts "
            "that are missing from the source."
        )
    }

    return bridge_name, bridge_node


def make_bridge_edge(source_node, bridge_node_name, bridge_node):
    added_high = bridge_node["added_high_level_concepts"]

    return {
        "from": source_node,
        "to": bridge_node_name,
        "edge_type": "bridge",
        "target_node": bridge_node["target_node"],
        "added_high_level_concepts": added_high,
        "missing_low_level_concepts_from_source": bridge_node[
            "missing_low_level_concepts_from_source"
        ],
        "label": "add " + ", ".join(added_high) if added_high else "same abstraction",
    }


MIT_NAMES = sorted(MIT_node_concepts.keys(), key=node_sort_key)
MOT_NAMES = sorted(MOT_node_concepts.keys(), key=node_sort_key)


# MIT -> MOT bridge nodes
for mit_node, mot_node in product(MIT_NAMES, MOT_NAMES):
    bridge_name, bridge_node = create_bridge_node(mit_node, mot_node)
    nodes[bridge_name] = bridge_node
    edges.append(make_bridge_edge(mit_node, bridge_name, bridge_node))


# MOT -> MIT bridge nodes
for mit_node, mot_node in product(MIT_NAMES, MOT_NAMES):
    bridge_name, bridge_node = create_bridge_node(mot_node, mit_node)
    nodes[bridge_name] = bridge_node
    edges.append(make_bridge_edge(mot_node, bridge_name, bridge_node))


In [76]:
# ============================================================
# Create physical PDDL folders/files for bridge nodes
# ============================================================
#
# Normal MIT/MOT node folders are generated from their CSV files above.
# Bridge nodes do not always have CSV files, so this cell creates:
#
#   OUTPUT_BASE_DIR / <bridge-node> / domain.pddl
#   OUTPUT_BASE_DIR / <bridge-node> / problem.pddl
#
# Example:
#   model/MIT3-MOT1/domain.pddl
#   model/MIT3-MOT1/problem.pddl

def merge_concept_mappings(*mappings):
    """
    Merge concept -> predicate-list dictionaries.

    If the same concept appears in multiple mappings, predicates are combined
    while preserving order.
    """
    merged = {}

    for mapping in mappings:
        for concept, predicates in mapping.items():
            merged.setdefault(concept, [])

            for pred in predicates:
                if pred not in merged[concept]:
                    merged[concept].append(pred)

    return merged


def predicates_for_bridge_node(node_info, concept_to_predicates):
    """
    Bridge rule:
      - keep source node's retained low-level predicates
      - add high-level concept predicates such as c1, c2, c11, if these
        predicates exist in the FULL domain/problem

    The high-level concepts are not expanded into their low-level predicates,
    because bridge nodes are meant to expose only abstract cross-domain context.
    """
    allowed = set()

    # Low-level predicates retained from the source node
    for concept in node_info.get("retained_low_level_concepts", []):
        allowed.update(concept_to_predicates.get(concept, []))

    # High-level bridge predicates, e.g., c1, c2, c7, c11
    for high_concept in node_info.get("retained_high_level_concepts", []):
        allowed.add(high_concept)

    return allowed


def materialize_bridge_nodes():
    full_domain_text = read_file(FULL_DOMAIN_FILE)
    full_problem_text = read_file(FULL_PROBLEM_FILE)

    concept_to_predicates = merge_concept_mappings(
        MIT_concept_mapping,
        MOT_concept_mapping
    )

    created = []

    for node_name, node_info in nodes.items():
        if node_info.get("node_type") != "bridge":
            continue

        output_dir = OUTPUT_BASE_DIR_PATH / node_name
        output_dir.mkdir(parents=True, exist_ok=True)

        output_domain_file = output_dir / "domain.pddl"
        output_problem_file = output_dir / "problem.pddl"

        allowed_predicates = predicates_for_bridge_node(
            node_info,
            concept_to_predicates
        )

        updated_domain = update_domain(full_domain_text, allowed_predicates)
        updated_problem = update_problem(full_problem_text, allowed_predicates)

        write_file(output_domain_file, updated_domain)
        write_file(output_problem_file, updated_problem)

        # Keep JSON paths consistent with the physical output directory.
        model_dir_name = Path(OUTPUT_BASE_DIR).name
        node_info["domain"] = f"{model_dir_name}/{node_name}/domain.pddl"
        node_info["problem"] = f"{model_dir_name}/{node_name}/problem.pddl"

        created.append(node_name)

        print("=" * 70)
        print(f"Created bridge node: {node_name}")
        print(f"Domain: {output_domain_file}")
        print(f"Problem: {output_problem_file}")
        print(f"Low-level concepts: {node_info.get('retained_low_level_concepts', [])}")
        print(f"High-level concepts: {node_info.get('retained_high_level_concepts', [])}")
        print(f"Predicates retained: {len(allowed_predicates)}")

    print("\nTotal bridge nodes created:", len(created))


materialize_bridge_nodes()

Created bridge node: MIT0-MOT0
Domain: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT0-MOT0/domain.pddl
Problem: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT0-MOT0/problem.pddl
Low-level concepts: ['C1', 'C2', 'C6', 'C7', 'C12']
High-level concepts: ['c11']
Predicates retained: 17
Created bridge node: MIT0-MOT1
Domain: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT0-MOT1/domain.pddl
Problem: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT0-MOT1/problem.pddl
Low-level concepts: ['C1', 'C2', 'C6', 'C7', 'C12']
High-level concepts: ['c11']
Predicates retained: 17
Created bridge node: MIT0-MOT2
Domain: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT0-MOT2/domain.pddl
Problem: /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/model/MIT0-MOT2/problem.pddl
Low-level concepts: ['C1', 'C2', 'C6', 'C7', 'C12']
High-level conc

In [77]:
# ============================================================
# Save lattice JSON
# ============================================================

concept_metadata_all = {
    c: concept_metadata(c)
    for c in FULL_CONCEPTS
}

high_level_concept_metadata_all = {
    to_high_level_concept(c): concept_metadata(c)
    for c in FULL_CONCEPTS
}

lattice = {
    "most_abstracted_node": MOST_ABSTRACTED_NODE,
    "full_node": FULL_NODE,

    "concept_universe": FULL_CONCEPTS,
    "high_level_concept_universe": ALL_HIGH_LEVEL_CONCEPTS,

    "concept_metadata": concept_metadata_all,
    "high_level_concept_metadata": high_level_concept_metadata_all,

    "nodes": nodes,
    "edges": edges,
}

with open(LATTICE_JSON_PATH, "w") as f:
    json.dump(lattice, f, indent=2)

print(f"Saved lattice JSON -> {LATTICE_JSON_PATH}")
print(json.dumps(lattice, indent=2))


Saved lattice JSON -> /content/drive/MyDrive/Colab Notebooks/EARG/test_lattice_generation/lattice_output/eval_lattice_with_bridge_nodes.json
{
  "most_abstracted_node": "M_ABS",
  "full_node": "FULL",
  "concept_universe": [
    "C1",
    "C2",
    "C6",
    "C7",
    "C11",
    "C12"
  ],
  "high_level_concept_universe": [
    "c1",
    "c2",
    "c6",
    "c7",
    "c11",
    "c12"
  ],
  "concept_metadata": {
    "C1": {
      "low_level_concept": "C1",
      "high_level_concept": "c1",
      "mitre_attack_ics_tactic": "Initial Access",
      "meaning": "The adversary is trying to get into the ICS environment."
    },
    "C2": {
      "low_level_concept": "C2",
      "high_level_concept": "c2",
      "mitre_attack_ics_tactic": "Execution",
      "meaning": "The adversary is trying to run code or manipulate system functions, parameters, and data in an unauthorized way."
    },
    "C6": {
      "low_level_concept": "C6",
      "high_level_concept": "c6",
      "mitre_attack_ics_tact